# Day 3 | Lab 3.3: Tool Calling Agents with LangChain (v1)

**Duration:** ~1.5 hours

**Scenario.** A user asks a multi-faceted question: "What's 3.14 × 2.718? What's the weather in Mumbai? And what are the 4 major Agentic AI design patterns?". A single chain with one model can't answer all three — we need tools (math, weather, web search) and an LLM that picks the right one for each part.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Use **built-in LangChain tools** (Tavily search) and inspect their schema.
2. Build **custom tools** with the `@tool` decorator (simple) and `StructuredTool` (with Pydantic-typed inputs).
3. Bind tools to a chat model and inspect the model's `tool_calls` output.
4. Construct a tool-calling agent with the LangChain v1 **`create_agent`** (replaces legacy `AgentExecutor`).
5. Debug tool-call failures — what to do when the agent picks the wrong tool, hits a tool exception, or loops.

**What we deliberately drop from the source notebook**
- Long Section 1 introduction and the Wikipedia tool demo (redundant — Tavily covers built-in-tool patterns).
- The `markitdown` web-extraction custom tool (heavy dependency, marginal pedagogical value).
- The non-native tool-calling section using prompt-based JSON (niche — every modern frontier model has native tool-calling).
- Manual tool-routing while-loops — replaced with `create_agent` (LangChain v1).

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-community>=0.3' 'langchain-tavily>=0.2' requests pydantic


## 2. API Key Configuration

This lab needs four keys: OpenAI (model), Tavily (web search), WeatherAPI (weather tool), and Groq (used for one comparison call).


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Verify keys are loaded (prints status, never prints actual values)
for key in ['OPENAI_API_KEY', 'TAVILY_API_KEY', 'WEATHER_API_KEY', 'GROQ_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Imports & LLM Init


In [ ]:
from langchain_openai import ChatOpenAI

# We use a non-reasoning model (gpt-4.1-mini) here — temperature=0 for deterministic tool selection.
# Reasoning models like gpt-5-mini also work for tool calling but spend extra tokens on thinking.
chatgpt = ChatOpenAI(model='gpt-4.1-mini', temperature=0)
print('LLM initialized:', chatgpt.model_name)


---

## 4. Built-in Tool — Tavily Web Search

LangChain ships with adapters for many web-search APIs. **Tavily** is purpose-built for LLM agents: it returns short structured snippets with full-content option, optimized for downstream prompts.


In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results=4,           # cap results for token budget
    search_depth='advanced', # 'basic' or 'advanced'
)

print('Tool name:', tavily_tool.name)
print('Description:', tavily_tool.description)
print('Args schema:', tavily_tool.args)


In [ ]:
# Direct invocation (without an LLM) — useful for testing
out = tavily_tool.invoke({'query': 'What are the major Agentic AI design patterns in 2026?'})
print(type(out).__name__)
if isinstance(out, dict) and 'results' in out:
    for r in out['results'][:2]:
        print('-', r.get('title', '')[:80])
        print(' ', (r.get('content') or '')[:200], '...')
else:
    print(str(out)[:400], '...')


---

## 5. Custom Tool with `@tool` — the Quick Form

The fastest way to define a tool: decorate any Python function. The function name becomes the tool name, the docstring becomes the description (the LLM reads it), the type hints become the schema.


In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the product."""
    return a * b

# Inspect the auto-generated schema
print('name        :', multiply.name)
print('description :', multiply.description)
print('args        :', multiply.args)


In [ ]:
# Direct invocation — the tool is just a Runnable under the hood
print(multiply.invoke({'a': 3.14, 'b': 2.718}))
print(multiply.invoke({'a': 7, 'b': 12}))


---

## 6. Custom Tool with `StructuredTool` + Pydantic — the Strict Form

When you need **explicit field descriptions** (which the LLM reads to pick the right tool) or **stricter validation**, use `StructuredTool.from_function(...)` with a Pydantic `args_schema`.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

class CalculatorInput(BaseModel):
    a: float = Field(description='The first number to multiply')
    b: float = Field(description='The second number to multiply')

def _multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

multiply_strict = StructuredTool.from_function(
    func=_multiply,
    name='multiply_strict',
    description='Use to multiply two numbers. Both arguments must be numeric.',
    args_schema=CalculatorInput,
)

print('name        :', multiply_strict.name)
print('description :', multiply_strict.description)
print('args (with field descriptions):')
for k, v in multiply_strict.args.items():
    print(f'  {k}: {v}')


In [ ]:
# Validation in action — passing a non-numeric raises pydantic.ValidationError
from pydantic import ValidationError

try:
    multiply_strict.invoke({'a': 3.14, 'b': 'abc'})
except ValidationError as e:
    print('Caught ValidationError:')
    print(e.errors()[0]['msg'])


---

## 7. Custom Tool — Real-World API Call (Weather)

A tool that calls an external HTTP API. Same `@tool` pattern — the LLM doesn't care that this one hits the network.


In [ ]:
import os
import requests
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> dict:
    """Get the current weather for a city or location. Returns a dict with temperature, condition, and humidity."""
    api_key = os.environ.get('WEATHER_API_KEY')
    if not api_key:
        return {'error': 'WEATHER_API_KEY not set in environment'}

    url = f'http://api.weatherapi.com/v1/current.json?key={api_key}&q={location}'
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        return {'error': f'API returned {response.status_code}'}

    data = response.json()
    if 'location' not in data:
        return {'error': 'Location not found'}
    return {
        'location': data['location']['name'],
        'country': data['location']['country'],
        'temp_c': data['current']['temp_c'],
        'condition': data['current']['condition']['text'],
        'humidity': data['current']['humidity'],
    }

print(get_weather.invoke({'location': 'Mumbai'}))


---

## 8. Tool Calling — Manual Form with `bind_tools`

Before reaching for an agent, it helps to see what tool-calling looks like at the **model API level**. `llm.bind_tools(tools)` returns a chat model that — when asked something tool-related — produces a `tool_calls` list instead of plain text.

This is the building block; `create_agent` (next section) wraps the loop of "call tool, feed result back to model, repeat" around it.


In [ ]:
tools = [tavily_tool, multiply, get_weather]

# Bind the tool definitions to the model — the LLM now sees their schemas
chatgpt_with_tools = chatgpt.bind_tools(tools)

user_query = (
    'I need three things from you: '
    '(1) what is 3.14 times 2.718, '
    '(2) the current weather in Mumbai, and '
    '(3) the four major Agentic AI design patterns in 2026.'
)

result = chatgpt_with_tools.invoke(user_query)
print('Model output content (often empty when tools are called):')
print(repr(result.content))
print(f'\nTool calls planned by the model: {len(result.tool_calls)}')


In [ ]:
# Inspect each planned tool call
for tc in result.tool_calls:
    print(f'• Tool: {tc["name"]}')
    print(f'  Args: {tc["args"]}')
    print()


In [ ]:
# Execute the planned tool calls manually — by hand, without an agent loop
toolkit = {t.name: t for t in tools}

for tc in result.tool_calls:
    tool_obj = toolkit[tc['name']]
    print(f'Calling {tc["name"]}({tc["args"]})...')
    output = tool_obj.invoke(tc['args'])
    output_str = str(output)
    print(f'  → {output_str[:200]}{"..." if len(output_str) > 200 else ""}')
    print()


### 📝 What the manual loop is missing

The cells above stop at "call the tools". They don't:
- Feed the tool *outputs* back to the model so it can produce a final user-facing answer.
- Handle the case where the model wants to call MORE tools after seeing the first batch's results.
- Limit iterations or detect tool-call loops.
- Stream events for a UI.

That's exactly what **`create_agent` (LangChain v1)** does for you.


---

## 9. The Modern Path — `create_agent` (LangChain v1)

Replaces the legacy `AgentExecutor` and the manual while-loop above. One function call gives you a runnable agent that:
1. Binds the tools to the LLM.
2. Runs the call → tool-execute → call-again loop until the model returns a final answer.
3. Caps iterations to prevent runaway loops.
4. Composes with `.invoke`, `.stream`, `.astream_events`, `.with_retry`, `.with_fallbacks` like any other Runnable.


In [ ]:
from langchain.agents import create_agent

system_prompt = (
    'You are a helpful assistant with access to tools for math, weather lookups, and web search. '
    'Use tools when they help; otherwise answer directly. '
    'When you have all the information, give the user a clear, concise final answer.'
)

agent = create_agent(
    chatgpt,
    tools=tools,
    system_prompt=system_prompt,
)


In [ ]:
# Invoke — input is a list of messages (or a dict with 'messages' key)
result = agent.invoke({
    'messages': [
        {'role': 'user', 'content': user_query},
    ]
})

# The final assistant message is the user-facing answer
final = result['messages'][-1]
print('=== Final answer ===')
print(final.content)


In [ ]:
# Inspect every message in the conversation — system, user, tool calls, tool results, final answer
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

for i, msg in enumerate(result['messages']):
    kind = type(msg).__name__
    if isinstance(msg, AIMessage) and msg.tool_calls:
        print(f'{i}. [{kind}] tool_calls: {[tc["name"] for tc in msg.tool_calls]}')
    elif isinstance(msg, ToolMessage):
        out = str(msg.content)[:150]
        print(f'{i}. [{kind}] from {msg.name}: {out}{"..." if len(str(msg.content)) > 150 else ""}')
    else:
        out = str(msg.content)[:150]
        print(f'{i}. [{kind}] {out}{"..." if len(str(msg.content)) > 150 else ""}')


---

## 10. Error Handling Inside an Agent Loop

What happens when a tool throws an exception? The default `create_agent` behaviour is to **catch the error and pass the message back to the model**, which then typically retries or asks the user for clarification. You can also customize this.


In [ ]:
# A tool that always fails — to test error handling
@tool
def flaky_tool(query: str) -> str:
    """Use this tool to test error handling."""
    raise RuntimeError('flaky_tool intentionally failed')

agent_with_flaky = create_agent(
    chatgpt,
    tools=[flaky_tool, multiply],
    system_prompt='You are an assistant. If a tool fails, gracefully fall back to your own knowledge or ask the user.',
)

result = agent_with_flaky.invoke({
    'messages': [{'role': 'user', 'content': 'Use flaky_tool to look up something, then tell me 5 + 5.'}]
})
print(result['messages'][-1].content)


---

## 11. Debugging — When the Agent Picks the Wrong Tool

When an agent misbehaves, the diagnosis usually comes from **reading the tool descriptions and `args_schema`** as if you were the LLM:

1. Are tool names distinct and descriptive? (`multiply` is clearer than `do_thing`.)
2. Do docstrings explain *when* to use the tool, not just *what* it does?
3. Are `Field(description=...)` strings filled in?
4. Are there overlapping tools? (Two web-search tools confuse the model.)
5. Is the system prompt giving a clear policy on tool use?

Bumping the model from `gpt-4.1-mini` to `gpt-5-mini` (a reasoning model) often fixes ambiguous tool selection at higher latency cost.


---

## 🔭 Forward Reference: Module 7 (LangGraph)

`create_agent` gives you a black-box loop that just works for ~80% of agentic use cases. When you need:
- **Explicit state** that survives across turns (typed via `TypedDict`)
- **Custom routing** between agent steps
- **Human-in-the-loop interrupts** mid-execution
- **Time-travel debugging** via checkpoints
- **Parallel sub-agents** (supervisor, swarm, hierarchical)

...you graduate to **LangGraph StateGraph** (Module 7). The same agent we built today will be re-implemented there with explicit state management — same tools, same model, but a graph instead of a loop.


---

## 12. Conclusion & Key Takeaways

| Concept | Walk-away |
|---|---|
| **Built-in tools** | LangChain ships with adapters (Tavily, Wikipedia, etc.) — drop-in, no setup |
| **`@tool` decorator** | Fastest way to define a custom tool — name, description, schema all derived from the function |
| **`StructuredTool` + Pydantic** | Use when you need explicit field descriptions or strict validation |
| **`bind_tools`** | Low-level — model produces a list of `tool_calls`; you execute them |
| **`create_agent` (LangChain v1)** | High-level — wraps the call→tool→call loop, handles errors, caps iterations |
| **Debugging tools** | Ambiguous tool selection usually = unclear name/description/`Field(description=...)` |

## 13. Stretch Exercise (Optional)
1. **Wikipedia tool back in.** Add `WikipediaQueryRun` from `langchain_community.tools` and observe how the agent picks Tavily vs Wikipedia for different query types.
2. **Iteration cap.** Set `max_iterations` (or wrap the agent with a manual counter) and force a query that requires more steps than the cap allows — observe the failure mode.
3. **Tool fallback.** Combine `.with_fallbacks()` to swap to a different model when the primary times out mid-tool-loop.
4. **Streaming events.** Use `agent.astream_events(...)` and print each `on_tool_start` / `on_tool_end` event as the agent runs.
5. **A second multiplier.** Add a `safe_multiply` tool that rejects negative numbers, then ask for `multiply(-3, 5)` — does the agent correctly pick the unsafe tool?


---

## Interview Preparation

---

**Q1. What is the difference between `create_agent` (v1) and `AgentExecutor` (legacy)?**

*Hint:* Runnable surface, simpler return shape, native composition.

*Answer sketch:* `AgentExecutor` was a class that wrapped a separate `Agent` + tool-running loop, returned a dict with intermediate steps, and didn't compose with LCEL. `create_agent` (LangChain v1) returns a Runnable — supports `.invoke`/`.stream`/`.astream_events`/`.with_retry`/`.with_fallbacks` natively, returns `{messages: [...]}`, and uses the same model `tool_calls` API instead of the older function-calling-via-prompt approach.

---

**Q2. How does the LLM "decide" to call a tool — what's actually in the prompt?**

*Hint:* Tool schemas are sent in a structured way, not just text.

*Answer sketch:* Modern providers (OpenAI, Anthropic, Gemini) accept a `tools` parameter alongside `messages` — each tool has a name, description, and JSON schema for arguments. The model decodes naturally to a `tool_calls` field instead of plain text when its policy says "a tool would help here". The decision is learned during fine-tuning; the prompt's job is to give clear tool descriptions and a system prompt explaining when to use them.

---

**Q3. What's the difference between `@tool` and `StructuredTool`, and when use each?**

*Hint:* Schema control.

*Answer sketch:* `@tool` derives the args schema from the function's type hints — fast, sufficient for ~80% of cases. `StructuredTool.from_function` lets you supply an explicit Pydantic `args_schema` with `Field(description=...)` strings, which the LLM reads to disambiguate similar tools. Use `@tool` first; reach for `StructuredTool` when the LLM keeps mis-routing or when fields need stricter validation.

---

**Q4. How do you handle a tool that throws an exception during the agent loop?**

*Hint:* Default behaviour + override.

*Answer sketch:* By default, `create_agent` catches the exception, attaches the error message to the `ToolMessage` returned to the model, and lets the model decide how to recover (often it falls back to its own knowledge or asks the user). For custom behaviour, wrap the tool with try/except in the function body, or set `handle_tool_errors=` on the agent.

---

**Q5. What does `StructuredTool.from_function` give you over `@tool`?**

*Hint:* Per-field metadata.

*Answer sketch:* Three things: (1) explicit Pydantic `args_schema` with `Field(description=...)` per argument — the LLM reads these to pick the right tool, (2) Pydantic validators (e.g., `gt=0`, `pattern=r'...'`) that block invalid arguments before the function runs, (3) a separate name/description from the function name/docstring.

---

**Q6. When would you choose LangChain agents over a hand-rolled while-loop?**

*Hint:* What the loop has to handle.

*Answer sketch:* Always, for production. The hand-rolled loop has to handle iteration limits, tool-error recovery, message-history threading, parallel tool calls, streaming, retries, and provider-portable schemas. `create_agent` does all of this and composes with the rest of LCEL. Roll your own only when you specifically need state/control LangGraph doesn't yet expose (rare).

---

**Q7. How do you limit the number of tool-call iterations?**

*Hint:* Configure on the agent.

*Answer sketch:* Pass `max_iterations` (or its v1 equivalent — `recursion_limit` on the underlying graph config) to `create_agent` or to `.invoke(..., config={'recursion_limit': 25})`. Without a cap, a misbehaving model can loop forever calling the same tool. Set a sensible upper bound based on the task — typically 10–20 for production.

---

**Q8. How do you debug when the agent picks the wrong tool?**

*Hint:* Read your tool definitions as if you were the LLM.

*Answer sketch:* Walk the diagnosis: (1) tool name distinct and descriptive? (2) docstring explains *when* to use? (3) `Field(description=...)` filled in? (4) overlapping tools? (5) clear system prompt policy? Bumping to a reasoning model (`gpt-5-mini`, `o1`) often fixes ambiguous selection at higher latency cost. Use `astream_events` to see the model's intermediate `on_tool_start` events and the args it's passing.
